# PKM_RF Triage Environment Setup & Package Installer (`setup.ipynb`)

Welcome to the **PKM_RF Emergency Severity Index (ESI) Triage Machine Learning System**.
This notebook initializes your environment, automatically installs all required Python and R packages (including `lightgbm`, `caret`, `optuna`, `pROC`, `e1071`), validates project configuration, checks dataset availability, creates output directories, and provides execution triggers for all pipeline notebooks.

--- 
### Project Architecture & Notebook Inventory

1. **Configuration**: [`config/triage_conf.json`](file:///home/apt2736/PKM_RF/config/triage_conf.json)
   - Central project configuration defining dataset paths, target column (`esi`), train/val/test split ratios, random seed, and resampling ratios.

2. **Engineered Feature Distribution Plots**: [`models/plot_engineered.ipynb`](file:///home/apt2736/PKM_RF/models/plot_engineered.ipynb)
   - Generates distribution graphs for 26 feature-engineered inputs across ESI classes `1` through `5` in `plots/engineered_feature_distributions/`.

3. **3-Tier Hierarchical Training**: [`models/train_hierarchical_lightgbm_layers.ipynb`](file:///home/apt2736/PKM_RF/models/train_hierarchical_lightgbm_layers.ipynb)
   - Trains 4 LightGBM sub-models with layer-exclusive features (`deploy/*.rds`).

4. **Combined Master Pipeline Benchmark**: [`models/combined_hierarchical_triage_pipeline.ipynb`](file:///home/apt2736/PKM_RF/models/combined_hierarchical_triage_pipeline.ipynb)
   - Evaluates Soft Probabilistic Joint Product predictions vs Hard Case-Selector routing on the 1% holdout test set.

5. **Native C Transpilation**: [`models/transpile_soft_pipeline_to_c.ipynb`](file:///home/apt2736/PKM_RF/models/transpile_soft_pipeline_to_c.ipynb)
   - Transpiles all sub-models into zero-dependency C code (`deploy/triage_soft_pipeline.c`).

6. **Layer 1 Anomaly Detection Benchmark**: [`models/benchmark_layer1_anomaly_detectors.ipynb`](file:///home/apt2736/PKM_RF/models/benchmark_layer1_anomaly_detectors.ipynb)
   - Benchmarks One-Class SVM, Isolation Forest, K-Means, and LightGBM baseline.

7. **Automated Resampling Calibration**: [`models/calibrate_resampling_ratios_5fold_cv.ipynb`](file:///home/apt2736/PKM_RF/models/calibrate_resampling_ratios_5fold_cv.ipynb)
   - Optimizes Layer 1 class weight multipliers and Layer 2 resampling ratios using 5-Fold Stratified CV with Optuna.

In [ ]:
# ---------------------------------------------------------
# Step 1: Python & System Tool Installation (cmake, lightgbm, rpy2, etc.)
# ---------------------------------------------------------
import sys
import subprocess
import os
import json
print("=== Step 1: Python & System Tool Verification ===")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Working Dir:    {os.getcwd()}")
required_py_pkgs = ['cmake', 'numpy', 'pandas', 'scipy', 'sklearn', 'lightgbm', 'optuna', 'matplotlib', 'seaborn', 'rpy2']
missing_py = []
for pkg in required_py_pkgs:
    try:
        __import__(pkg)
        print(f"  [OK] Python package: {pkg}")
    except ImportError:
        missing_py.append(pkg)
        print(f"  [MISSING] Python package: {pkg}")
if missing_py:
    print(f"\nInstalling missing Python packages: {missing_py} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_py)
    print("Python packages installed successfully!")
# Ensure output directories exist
for folder in ['deploy', 'reports', 'plots', 'plots/engineered_feature_distributions']:
    os.makedirs(folder, exist_ok=True)
    print(f"  [DIR OK] {folder}/")

In [ ]:
# ---------------------------------------------------------
# Step 2: Validate Project Configuration & Data Source
# ---------------------------------------------------------
config_file = "config/triage_conf.json"
if os.path.exists(config_file):
    with open(config_file, "r") as f:
        config = json.load(f)
    print("\n=== Step 2: Project Configuration Validated ===")
    print(f"  Data Source: {config['path']['data_source']}")
    print(f"  Target Col:  {config['classes']['target_col']}")
    
    data_path = config['path']['data_source']
    if os.path.exists(data_path):
        print(f"  [DATASET OK] Found data file: {data_path}")
    else:
        print(f"  [WARNING] Data file not found at: {data_path}")
else:
    print(f"[ERROR] Configuration file missing: {config_file}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Initialize rpy2 for R Execution
# ---------------------------------------------------------
try:
    %load_ext rpy2.ipython
    print("rpy2 extension loaded successfully.")
except Exception as e:
    print("Note on rpy2 initialization:", e)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Full R Package Installation & Verification (lightgbm, caret, pROC, etc.)
# ---------------------------------------------------------
# Prefer Posit PPM Linux Binaries for fast pre-compiled package installation
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
r_pkgs <- c("jsonlite", "dplyr", "ggplot2", "tidyr", "pROC", "e1071", "caret", "lightgbm")
cat("=== Step 4: Auditing & Installing R Packages ===\n")
for (pkg in r_pkgs) {
  if (!suppressWarnings(require(pkg, character.only = TRUE, quietly = TRUE))) {
    cat(sprintf("  [INSTALLING] R package: %s ...\n", pkg))
    tryCatch({
      install.packages(pkg, dependencies = TRUE)
    }, error = function(e) {
      cat(sprintf("  [WARNING] Primary repo install failed for %s: %s. Trying CRAN fallback...\n", pkg, e$message))
      install.packages(pkg, repos = "https://cloud.r-project.org", dependencies = TRUE)
    })
  } else {
    cat(sprintf("  [OK] R package: %s\n", pkg))
  }
}
# Special Verification for lightgbm in R
if (!suppressWarnings(require("lightgbm", quietly = TRUE))) {
  cat("\n  [RETRYING] Installing lightgbm from CRAN fallback...\n")
  install.packages("lightgbm", repos = "https://cloud.r-project.org")
}
cat("\n=== Final R Package Audit Summary ===\n")
all_ok <- TRUE
for (pkg in r_pkgs) {
  is_loaded <- suppressWarnings(require(pkg, character.only = TRUE, quietly = TRUE))
  if (is_loaded) {
    cat(sprintf("  [SUCCESS] %s is installed and ready.\n", pkg))
  } else {
    cat(sprintf("  [FAILED] %s could not be loaded.\n", pkg))
    all_ok <- FALSE
  }
}
if (all_ok) {
  cat("\nALL R DEPENDENCIES SUCCESSFULLY INSTALLED AND VERIFIED!\n")
} else {
  cat("\nWARNING: Some R packages failed to install. Please check error logs above.\n")
}

In [ ]:
# ---------------------------------------------------------
# Step 5: Pipeline Execution Triggers
# ---------------------------------------------------------
print("SETUP FINISHED: All Python & R packages (including lightgbm and caret) have been verified!")